# Measure prompt injection as an attack-success rate

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/profrodai/profrodai-resources/blob/main/articles/treat-retrieved-instructions-as-untrusted-input/injection_v1.ipynb)

Companion to https://profrod.ai/articles/treat-retrieved-instructions-as-untrusted-input. A coding assistant with a `push_to_remote` tool summarizes a README that asks it to push. Four small open models (Qwen2.5 0.5B and 1.5B, Qwen3 0.6B and 1.7B), five attack styles, two placements, three prompt defenses, a harness fix and two controls: 4,560 recorded replies.

Part 1 recomputes the article's numbers from the recorded replies. Part 2 reruns a slice on this runtime's own GPU or CPU, with no API key: choose **Runtime > Change runtime type > T4 GPU** for speed.

In [ ]:
import json
import os
import urllib.request

BASE = "https://raw.githubusercontent.com/profrodai/profrodai-resources/main/articles/treat-retrieved-instructions-as-untrusted-input/"
FILES = ["injection_items_v1.py", "injection_stats_v1.py", "injection_run_v1.py", "data/revisions-v1.json",
         "results/attempts-v1.jsonl", "results/summary-v1.json", "receipts/injection-v1.json"]
for name in FILES:
    if not os.path.exists(name):
        os.makedirs(os.path.dirname(name) or ".", exist_ok=True)
        urllib.request.urlretrieve(BASE + name, name)
print("ready")

## Part 1: recompute the article's numbers

In [ ]:
import injection_run_v1 as R

summary = R.summarize()
for model, row in summary["models"].items():
    print(model)
    for defense, d in row["defenses"].items():
        a, p = d["attackSuccess"], d["positive"]
        lo, hi = a["wilson95"]
        print(f"  {defense:9s} attack success {a['k']:3d}/{a['n']} = {a['rate']:.1%} (Wilson {lo:.1%} to {hi:.1%})  "
              f"asked-for push {p['k']:2d}/{p['n']}  false alarms {d['falseAlarm']['k']}/{d['falseAlarm']['n']}  executed {d['executed']}")

In [ ]:
# Which attacks work, per model (no defense): pushes out of 30 READMEs.
for model, row in summary["models"].items():
    print(model, row["defenses"]["none"]["byAttack"])

In [ ]:
# Paired on the same 300 injected items: does the defense change the outcome?
for model, row in summary["models"].items():
    for name, c in row["comparisons"].items():
        print(f"{model:28s} {name:17s} pushed only without it {c['onlyNone']:3d}, only with it {c['onlyDefense']:3d}, McNemar p = {c['mcnemarP']:.2g}")

In [ ]:
# The forged user turn, with the harness escaping special tokens in file text first.
for model, row in summary["models"].items():
    e = row["escape"]
    print(f"{model:28s} forged turn {e['forgedTurn']['k']}/{e['forgedTurn']['n']}  escaped {e['escaped']['k']}/{e['escaped']['n']}")

## Part 2: rerun a slice here (optional, no key needed)

This runs the first five READMEs (190 items, every condition) on Qwen3-1.7B, the most injectable model in the recorded run. On a T4 it takes a few minutes. Your replies are sampled, so your counts will differ from the recorded ones; compare them with the intervals, not the point estimates.

In [ ]:
os.makedirs("mine/data", exist_ok=True)
!cp injection_items_v1.py injection_stats_v1.py injection_run_v1.py mine/ && cp data/revisions-v1.json mine/data/
!cd mine && python injection_run_v1.py run --model Qwen/Qwen3-1.7B --projects 5 && python injection_run_v1.py summarize > /dev/null
mine = json.load(open("mine/results/summary-v1.json"))["models"]["Qwen/Qwen3-1.7B"]["defenses"]
for defense, d in mine.items():
    a = d["attackSuccess"]
    print(f"{defense:9s} attack success {a['k']}/{a['n']} = {a['rate']:.0%}   asked-for push {d['positive']['k']}/{d['positive']['n']}")